# BCU accumulator tests — Q5 (int64 fixed-point) & Q6 (two's-complement wrap)

Scope: **only** algo-table items **Q5** and **Q6**.

* **Q5** — the accumulator is now `int64` fixed-point. Integer math is exact by
  construction: `prod_mant` (<= 14 significant bits) scaled by an integer
  exponent via bit shifts (`<<` / arithmetic `>>` == floor), no float
  rounding anywhere on the accumulation path.
* **Q6** — overflow policy is **two's-complement wrap-and-continue** at
  `acc_bit_width` in `{46, 48, 54, 56}` after every term (replaces the old
  freeze + clamp).

Run all cells top to bottom; every assertion passing = tests pass.
Quantizers, TRU and PCU are untouched by Q5/Q6 and are not tested here (the
main demo notebook remains the end-to-end regression suite).


In [1]:
import os
import sys
# make npe_mx_emu importable whether jupyter runs from the repo root or from EL_nb/
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import torch

from npe_mx_emu import (
    SUPPORTED_ACC_WIDTHS,
    MXConv2D_Loops,
    bcu,
    wrap_accumulator,
)

print("supported acc widths:", SUPPORTED_ACC_WIDTHS)


supported acc widths: (46, 48, 54, 56)


## Pure-python exact reference

The oracle: integer arithmetic (python ints, unbounded), arithmetic right shift
(== floor) for negative exponents, and mod `2^N` two's-complement wrap after
**every** term — same order of operations as `bcu`.


In [2]:
def ref_bcu_python(x_data, x_exp, w_data, w_exp, acc_bit_width):
    """Exact integer reference for bcu: K rows x L columns, shared weights per column."""
    half = 1 << (acc_bit_width - 1)
    mod  = 1 << acc_bit_width
    out = []
    for k in range(x_data.shape[-2]):
        s = 0
        for l in range(x_data.shape[-1]):
            m = int(x_data[k][l]) * int(w_data[l])
            e = int(x_exp[k][l]) + int(w_exp[l])
            term = (m << e) if e >= 0 else (m >> (-e))  # python >> is arithmetic (floor)
            s = s + term
            s = ((s + half) % mod) - half               # two's-complement wrap per add
        out.append(s)
    return out


## Q5 — int64 fixed-point accumulation is exact

**Test A**: randomized mantissas/exponents (incl. negatives and negative
shifts), magnitudes below saturation. Two checks: (i) `bcu` must equal the
python integer oracle **bit-exactly**; (ii) the old formula recomputed in
float64 (exact for integers < 2^53) must agree with the int64 path — proof
the integer shifts reproduce `exp2()*floor` term-for-term. Plain float32
accumulation is **not** valid even here (sums can exceed 2^24): that is
exactly what Test B demonstrates.

**Test B**: the float32 wall made concrete — accumulate `2^25`, then two `+1` terms.
float32's ulp at `2^25` is `2`, so each `+1` sits exactly between two
representables and ties-to-even drops it: the low bits silently vanish
(delta `−2`). The int64 path must keep them, bit-exact vs the oracle.


In [3]:
# ---- Q5 / Test A: small-magnitude randomized round-trips -------------------
torch.manual_seed(7)

for trial, (K, L) in enumerate([(1, 32), (4, 32), (2, 96), (3, 8)]):
    x_data = torch.randint(-127, 128, (K, L), dtype=torch.int32)
    w_data = torch.randint(-127, 128, (L,),   dtype=torch.int32)
    x_exp  = torch.randint(-4, 8, (K, L),     dtype=torch.int32)
    w_exp  = torch.randint(0, 8, (L,),        dtype=torch.int32)

    out = bcu(x_data, x_exp, w_data, w_exp, acc_bit_width=48)
    ref = ref_bcu_python(x_data, x_exp, w_data, w_exp, 48)

    assert out.dtype == torch.int64, f"Q5: accumulator must be int64, got {out.dtype}"
    assert [int(v) for v in out] == ref, f"Q5: trial {trial} mismatch"

    # float64 reproduction of the old formula is exact for these magnitudes (< 2^53)
    # -> proves the integer shifts reproduce exp2()*floor term-for-term
    prod = (x_data * w_data).double()
    shifted_f = (prod * torch.exp2((x_exp + w_exp).double())).floor()
    old_path = shifted_f.sum(dim=-1)
    assert torch.equal(out, old_path.long()), f"Q5: trial {trial} formula mismatch"

print("Q5 Test A OK: int64 bcu == python integer oracle == old formula in float64")


Q5 Test A OK: int64 bcu == python integer oracle == old formula in float64


In [4]:
# ---- Q5 / Test B: float32 loses low bits once the accumulator exceeds 2^24
# terms: 2^25 then two +1s. float32 ulp at 2^25 is 2 -> 2^25 + 1 sits exactly
# between two representables and ties-to-even drops it back: the +1s vanish.
K, L = 1, 3
x_data = torch.tensor([[64, 1, 1]], dtype=torch.int32)
w_data = torch.tensor([256, 1, 1],  dtype=torch.int32)
x_exp  = torch.tensor([[11, 0, 0]], dtype=torch.int32)
w_exp  = torch.zeros((3,),          dtype=torch.int32)

terms     = [64 * 256 << 11, 1, 1]   # [2^25, 1, 1]
exact_sum = (64 * 256 << 11) + 2     # 2^25 + 2 = 33554434

out = bcu(x_data, x_exp, w_data, w_exp, acc_bit_width=54)  # 54-bit: no wrap here
assert int(out[0]) == exact_sum, "Q5: int64 accumulation not exact"
ref = ref_bcu_python(x_data, x_exp, w_data, w_exp, 54)
assert ref == [exact_sum], "Q5: bcu != python integer oracle"

# reproduce the old float32 accumulator (pre-Q5) on the same terms
s32 = torch.tensor(0.0)  # float32
for t in terms:
    s32 = s32 + torch.tensor(float(t))
print(f"exact int sum   : {exact_sum}")
print(f"old float32 sum : {int(s32.item())}  (delta {int(s32.item()) - exact_sum})")
assert int(s32.item()) != exact_sum, "sanity: this case must exercise float32 rounding"
print("Q5 Test B OK: int64 keeps the low bits that float32 rounds away")


exact int sum   : 33554434
old float32 sum : 33554432  (delta -2)
Q5 Test B OK: int64 keeps the low bits that float32 rounds away


## Q6 — two's-complement wrap at `acc_bit_width`

**Test C**: `wrap_accumulator` unit checks on known values (in-range values
untouched; positive overflow wraps negative; negative stays negative).

**Test D**: full `bcu` wrap behavior per width in `{46, 48, 54, 56}`:
results equal the modular oracle; wrap is width-dependent; accumulation
**continues** from the wrapped state (no freeze, no clamp); negative side
wraps to positive; unsupported width raises.

Note on the witness term: a wrapped single term `m << e` is a function of the
residue `m mod 2^(N-e)` at width `N` — pathological mantissas (e.g.
`2^14 - 255`) can collide pairwise across widths, so Test D uses
`45 * 57 = 2565` whose four residues are all distinct.


In [5]:
# ---- Q6 / Test C: wrap_accumulator mechanics --------------------------------
def w(v, bits):
    return int(wrap_accumulator(torch.tensor(v, dtype=torch.int64), bits))

# in-range values pass through untouched (all widths)
for v in [0, 1, -1, 12345, -12345, (1 << 44), -(1 << 44)]:
    for b in SUPPORTED_ACC_WIDTHS:
        assert w(v, b) == v, f"Q6: in-range {v} changed at width {b}"

# bit (N-1) set  ->  exactly -2^(N-1)
for b in SUPPORTED_ACC_WIDTHS:
    assert w(1 << (b - 1), b) == -(1 << (b - 1)), f"Q6: sign-bit wrap wrong at {b}"

# positive overflow wraps to NEGATIVE (sign read from wrapped bits, not input)
assert w((1 << 47) + 5, 48) == -(1 << 47) + 5          # 2^47 + 5  ->  -2^47 + 5
assert w((1 << 45), 46) == -(1 << 45)                  # 2^45      ->  -2^45
assert w((1 << 55) - 1, 56) == (1 << 55) - 1           # top in-range positive, no wrap
assert w(-3, 48) == -3                                 # negatives pass through
assert w(-(1 << 47), 48) == -(1 << 47)                 # most-negative is representable

# unsupported width raises
for bad in (40, 24, 64):
    try:
        w(0, bad)
        raise AssertionError(f"Q6: width {bad} should have raised")
    except NotImplementedError:
        pass

print("Q6 Test C OK: wrap_accumulator mechanics (in-range, sign-bit, overflow, width guard)")


Q6 Test C OK: wrap_accumulator mechanics (in-range, sign-bit, overflow, width guard)


In [6]:
# ---- Q6 / Test D: bcu wrap behavior per width -------------------------------
# single overflowing term: 2565 << 44 (~2^55.6) overflows every supported width.
# witness chosen so its residues m mod 2^(N-44) = (1, 5, 517, 2565) are distinct
# per width -> four distinct wrap images. (m = 16129 = 2^14-255 would collide
# pairwise: +-2^44 and -255*2^44 each appear twice.)
K, L = 1, 3
x_data = torch.tensor([[45, 0, 0]], dtype=torch.int32)
w_data = torch.tensor([57, 5, 7],  dtype=torch.int32)
x_exp  = torch.tensor([[44, 0, 0]], dtype=torch.int32)
w_exp  = torch.zeros((L,),          dtype=torch.int32)

results = {}
for b in SUPPORTED_ACC_WIDTHS:
    out = bcu(x_data, x_exp, w_data, w_exp, acc_bit_width=b)
    ref = ref_bcu_python(x_data, x_exp, w_data, w_exp, b)
    assert [int(v) for v in out] == ref, f"Q6: bcu wrap != oracle at width {b}"
    results[b] = int(out[0])
    print(f"width {b}: wrapped p_sum = {results[b]}")

# wrap must be width-dependent (all four overflow here, each differently)
assert len(set(results.values())) == len(SUPPORTED_ACC_WIDTHS), "Q6: wrap not width-dependent"

# smaller exponent (2565 << 30 ~ 2^41.3) is in range for ALL widths -> identical
x_exp_in = torch.tensor([[30, 0, 0]], dtype=torch.int32)
outs = {b: int(bcu(x_data, x_exp_in, w_data, w_exp, acc_bit_width=b)[0]) for b in SUPPORTED_ACC_WIDTHS}
assert len(set(outs.values())) == 1, "Q6: in-range result must not depend on width"
print("in-range check: p_sum =", list(outs.values())[0], "(identical for all widths)")

# wrap-and-CONTINUE: giant positive term wraps negative, the next term is still added
x_data2 = torch.tensor([[127, 127, 3]], dtype=torch.int32)
w_data2 = torch.tensor([127, 127, 3],  dtype=torch.int32)
x_exp2  = torch.tensor([[44, 44, 0]],  dtype=torch.int32)
w_exp2  = torch.zeros((3,),            dtype=torch.int32)
for b in SUPPORTED_ACC_WIDTHS:
    out = bcu(x_data2, x_exp2, w_data2, w_exp2, acc_bit_width=b)
    ref = ref_bcu_python(x_data2, x_exp2, w_data2, w_exp2, b)
    assert [int(v) for v in out] == ref, f"Q6: wrap-and-continue mismatch at {b}"

# negative side wraps to positive
x_neg = torch.tensor([[-127, -127, 0]], dtype=torch.int32)
w_neg = torch.tensor([127, 127, 1],    dtype=torch.int32)
e_neg = torch.tensor([[44, 44, 0]],    dtype=torch.int32)
for b in SUPPORTED_ACC_WIDTHS:
    out = bcu(x_neg, e_neg, w_neg, torch.zeros((3,), dtype=torch.int32), acc_bit_width=b)
    ref = ref_bcu_python(x_neg, e_neg, w_neg, torch.zeros((3,), dtype=torch.int32), b)
    assert [int(v) for v in out] == ref, f"Q6: negative-wrap mismatch at {b}"

# unsupported width raises on both entry points
for bad in (40, 24, 64):
    try:
        bcu(x_data, x_exp, w_data, w_exp, acc_bit_width=bad)
        raise AssertionError(f"Q6: bcu width {bad} should have raised")
    except NotImplementedError:
        pass
try:
    MXConv2D_Loops(1, 1, 3, acc_bit_width=40)
    raise AssertionError("Q6: layer width 40 should have raised")
except NotImplementedError:
    pass
MXConv2D_Loops(1, 1, 3, acc_bit_width=48)  # supported width constructs fine

print("Q6 Test D OK: wrap per width, wrap-and-continue, negative wrap, width guards")


width 46: wrapped p_sum = 17592186044416
width 48: wrapped p_sum = 87960930222080
width 54: wrapped p_sum = -8919238324518912
width 56: wrapped p_sum = -26933636834000896
in-range check: p_sum = 2754147778560 (identical for all widths)
Q6 Test D OK: wrap per width, wrap-and-continue, negative wrap, width guards


In [7]:
print("=" * 60)
print("ALL Q5 / Q6 TESTS PASSED")
print("  Q5: int64 fixed-point accumulator, bit-exact vs integer oracle")
print("  Q6: two's-complement wrap-and-continue at", SUPPORTED_ACC_WIDTHS)


ALL Q5 / Q6 TESTS PASSED
  Q5: int64 fixed-point accumulator, bit-exact vs integer oracle
  Q6: two's-complement wrap-and-continue at (46, 48, 54, 56)
